In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import glob
import regex
from typing import Dict, List, Tuple, Union

import tqdm.notebook as tqdm

import numpy as np
import math
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import random_split

import librosa
import torchaudio

from scipy.io import wavfile
from sklearn.decomposition import PCA

import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import display, Audio, Markdown

%matplotlib inline
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')

In [4]:
sections = ["EEG"]

This is NOT the same class, some changes were made

In [34]:
class EEGDataset(Dataset):
    def __init__(self, path: str, audio_maps: dict, fragment_length: int = 2012,
                 partition_size: int = 32, sample_rate: int = 44100, sound_channel: int = 1,
                 preloaded_audios: bool = False, val_ratio: float = 0.15):
        '''
        path: path to sections (folders)
        audio_maps: two-level map: section names -> labels -> audio_paths
        fragment_lengtht: length of fragment after label
        partition_size: number of nonzero labels in each csv file
        sample_rate: audios' SR
        sound_channel: mono audio channel
        preloaded_audios: if audio_maps lead to paths (False) or directly to data like raw signal or FT (True)
        val_ratio: float in range [0, 1], N_val / N
        '''
        super().__init__()
        self.sections = sorted(os.listdir(path))
        assert set(self.sections) == set(audio_maps.keys()), "Sections must be the same!"
        self.audio_maps = audio_maps 
        self.preloaded_audios = preloaded_audios
        
        all_paths = [[os.path.join(path, sec, file) for file in sorted(os.listdir(os.path.join(path, sec)))] for sec in self.sections]
        num_all_files = [len(elem) for elem in all_paths]
        splits = [int(elem * val_ratio) for elem in num_all_files]
        
        self.val_paths = [sec_paths[:split] for sec_paths, split in zip(all_paths, splits)]
        self.paths = [sec_paths[split:] for sec_paths, split in zip(all_paths, splits)]
        
        self.sec_num_files = [len(elem) for elem in self.paths]
        self.sec_cumnum = np.cumsum(self.sec_num_files) * partition_size
        self.total_num_files = sum(self.sec_num_files)
        
        self.sec_num_val_files = [len(elem) for elem in self.val_paths]
        self.sec_val_cumnum = np.cumsum(self.sec_num_val_files) * partition_size
        self.total_num_val_files = sum(self.sec_num_val_files)
        
        self.partition_size = partition_size
        self.fragment_length = fragment_length
        self.sr = sample_rate
        self.sound_channel = sound_channel
        self.val_mode = False
        
    def __len__(self) -> int:
        num = self.total_num_val_files if self.val_mode else self.total_num_files
        return num * self.partition_size
    
    def set_val_mode(self, mode: bool):
        '''
        Switch between train/val subsets
        mode: 0 -> train, 1 -> val
        '''
        assert mode in [True, False], "Incorrect mode type!"
        self.val_mode = mode
        return self
    
    def to_section(self, idx: int) -> Tuple[int, int]:
        '''
        Get file section and inner index by its absolute index
        idx: absolute index
        return inner index, section number (idx)
        '''
        cumnum = self.sec_val_cumnum if self.val_mode else self.sec_cumnum
        section = np.where(idx < cumnum)[0][0]
        section_idx = idx if (section == 0) else (idx - cumnum[section - 1])
        return section, section_idx
    
    def get_audio(self, section: str, label: int) -> torch.Tensor:
        '''
        Get audio by section and corresponding label
        section: number of section
        label: one of label values in given section
        return: the audio
        '''
        section_name = self.sections[section]
        
        if self.preloaded_audios:
            return self.audio_maps[section_name][label]

        audio = pd.read_csv(self.audio_maps[section_name][label]).to_numpy()
        return torch.tensor(audio, dtype=torch.float32)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        '''
        int idx: file ID
        return: EEG fragment with its corresponding audio
        '''
        section, section_idx = self.to_section(idx)
        paths_source = self.val_paths if self.val_mode else self.paths
        file_path = paths_source[section][section_idx // self.partition_size]
        
        start = (section_idx % self.partition_size) * self.fragment_length
        end = start + self.fragment_length
        
        data = pd.read_feather(file_path).to_numpy()
        x, label = torch.tensor(data[start:end, 1:], dtype=torch.float32), data[start, 0].astype(int)
        
        audio = self.get_audio(section, label)
        
        return x, audio

In [35]:
base = os.path.join("Data", "Vartanov", "audios")

vniz = "vniz.csv"
vverh = "vverh.csv"
vpravo = "vpravo.csv"
vlevo = "vlevo.csv"
nazad = "nazad.csv"
vpered = "vpered.csv"
bezhat = "bezhat.csv"
stoyat = "stoyat.csv"

EEG_labels = {
    11: os.path.join(base, "EEG", vniz),
    21: os.path.join(base, "EEG", vniz),
    12: os.path.join(base, "EEG", vverh),
    22: os.path.join(base, "EEG", vverh),
    13: os.path.join(base, "EEG", vpravo),
    23: os.path.join(base, "EEG", vpravo),
    14: os.path.join(base, "EEG", vlevo),
    24: os.path.join(base, "EEG", vlevo),
    15: os.path.join(base, "EEG", nazad),
    25: os.path.join(base, "EEG", nazad),
    16: os.path.join(base, "EEG", vpered),
    26: os.path.join(base, "EEG", vpered),
    17: os.path.join(base, "EEG", bezhat),
    27: os.path.join(base, "EEG", bezhat),
    18: os.path.join(base, "EEG", stoyat),
    28: os.path.join(base, "EEG", stoyat),
}

audio_map = {
    "EEG": EEG_labels
}

In [36]:
dataset = EEGDataset("./Data/Vartanov/feather", audio_map)

WARNING: audio in new data is 15 PCA components of STFT, not raw wav!

In [41]:
EEG, audio = dataset[0]
EEG.shape, audio.shape

(torch.Size([2012, 63]), torch.Size([345, 15]))